In [8]:
import sys, site, platform
print("Python exe:", sys.executable)
print("Version:", sys.version)
print("Platform:", platform.platform())
print("Site-packages:", site.getsitepackages() if hasattr(site, "getsitepackages") else "n/a")
print("User site:", site.getusersitepackages())

Python exe: c:\Users\brobi\OneDrive\Desktop\Algo1\.venv\Scripts\python.exe
Version: 3.11.0 (main, Oct 24 2022, 18:26:48) [MSC v.1933 64 bit (AMD64)]
Platform: Windows-10-10.0.26100-SP0
Site-packages: ['c:\\Users\\brobi\\OneDrive\\Desktop\\Algo1\\.venv', 'c:\\Users\\brobi\\OneDrive\\Desktop\\Algo1\\.venv\\Lib\\site-packages']
User site: C:\Users\brobi\AppData\Roaming\Python\Python311\site-packages


In [9]:
print('hello world')

import alpaca_trade_api as tradeapi
import pandas as pd
import datetime
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

from datetime import datetime, timedelta, timezone
import alpaca
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from alpaca.data.enums import Adjustment, DataFeed

from typing import List, Tuple
import os

import ta
import numpy as np

hello world


In [10]:
import os
from dotenv import load_dotenv, find_dotenv

# 1) Load .env if you’re using one (finds it even if the notebook is in a subfolder)
load_dotenv(find_dotenv())

# 2) Debug helper: see whether Python can read the vars (masked)
def _check(name):
    v = os.getenv(name)
    print(f"{name}: {'OK' if v else 'MISSING'}", (v[:4] + '…') if v else '')

_check("ALPACA_KEY_ID")
_check("ALPACA_SECRET_KEY")

ALPACA_KEY_ID: OK PKA6…
ALPACA_SECRET_KEY: OK SBg0…


In [11]:
# Set up Alpaca API
API_KEY = os.getenv("ALPACA_KEY_ID")
SECRET_KEY = os.getenv("ALPACA_SECRET_KEY")

if not API_KEY or not SECRET_KEY:
    raise RuntimeError(
        "Missing ALPACA_API_KEY_ID or ALPACA_API_SECRET_KEY. "
        "Confirm they’re set in your shell and restart your IDE/terminal.")
BASE_URL = "https://paper-api.alpaca.markets"
client = StockHistoricalDataClient(API_KEY, SECRET_KEY)

In [12]:

def _safe_end(end: datetime | None = None, safety_min: int = 20) -> datetime:
    """Ensure end time is at least 20 min behind now to avoid recent SIP restriction."""
    now = datetime.now(timezone.utc)
    return (end or now) - timedelta(minutes=safety_min)


def fetch_1m_bars(symbol: str, start: datetime, end: datetime) -> pd.DataFrame:
    """Fetch 1-minute bars from Alpaca IEX feed."""
    req = StockBarsRequest(
        symbol_or_symbols=[symbol],
        start=start,
        end=end,
        timeframe=TimeFrame.Minute,
        adjustment=Adjustment.RAW,
        feed=DataFeed.IEX
    )
    bars = client.get_stock_bars(req)
    df = bars.df.reset_index()
    return df[df["symbol"] == symbol] if "symbol" in df.columns else df


def resample_to_30s(df: pd.DataFrame) -> pd.DataFrame:
    """Convert 1-minute bars to 30-second bars using forward-fill approximation."""
    if df.empty:
        return df

    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df = df.set_index("timestamp")
    df_30s = df.resample("30S").ffill()

    # Volume handling — split evenly across the two 30s periods in each minute
    df_30s["volume"] = df_30s["volume"] / 2.0
    return df_30s[["open", "high", "low", "close", "volume"]]


def add_indicators_and_target(df: pd.DataFrame, threshold: float = 0.001) -> pd.DataFrame:
    """Add SMA, RSI, target, and market status."""
    out = df.copy()

    # Indicators
    out["SMA_10"] = out["close"].rolling(10).mean()
    out["SMA_50"] = out["close"].rolling(50).mean()
    out["RSI_14"] = ta.momentum.RSIIndicator(out["close"], window=14).rsi()

    # Target
    out["Future_Close"] = out["close"].shift(-1)
    out["Future_Change"] = (out["Future_Close"] - out["close"]) / out["close"]
    out["Target"] = np.sign(out["Future_Change"].where(
        out["Future_Change"].abs() >= threshold, 0.0
    )).astype(int)

    # Market status
    def market_flag(ts: pd.Timestamp) -> str:
        t = ts.tz_convert("US/Eastern").time()
        if t >= pd.Timestamp("04:00").time() and t < pd.Timestamp("09:30").time():
            return "pre-market"
        if t >= pd.Timestamp("09:30").time() and t < pd.Timestamp("16:00").time():
            return "regular"
        if t >= pd.Timestamp("16:00").time() and t < pd.Timestamp("20:00").time():
            return "after-hours"
        return "closed"

    out["market_status"] = out.index.map(market_flag)

    return out.dropna()


def build_symbol_csv(symbol: str, start_iso: str, end_iso: str | None = None,
                     out_dir: str = "./data", threshold: float = 0.001):
    os.makedirs(out_dir, exist_ok=True)
    csv_path = os.path.join(out_dir, f"{symbol}_30s.csv")

    # ✅ If file exists, load and skip pulling
    if os.path.exists(csv_path):
        print(f"✅ {csv_path} already exists. Skipping API fetch.")
        return pd.read_csv(csv_path, parse_dates=["timestamp"], index_col="timestamp")

    start = pd.Timestamp(start_iso, tz="UTC").to_pydatetime()
    end = pd.Timestamp(end_iso, tz="UTC").to_pydatetime() if end_iso else None
    end = _safe_end(end)

    print(f"📥 Fetching 1-min IEX data for {symbol}...")
    raw = fetch_1m_bars(symbol, start, end)
    if raw.empty:
        print(f"⚠️ No data for {symbol}")
        return

    print(f"⏳ Resampling {symbol} to 30-second bars...")
    bars_30s = resample_to_30s(raw)

    print(f"📈 Adding indicators and target for {symbol}...")
    final_df = add_indicators_and_target(bars_30s, threshold)

    final_df.to_csv(csv_path, index=True)
    print(f"💾 Saved {symbol} → {csv_path}")
    return final_df


In [ ]:

# ---------------------------
# Run for multiple symbols
# ---------------------------
symbols = [
    
    "IBM","RGTI","QBTS","QUBT",'IONQ', "QS", "AMD",    "SLDP",    "MSFT",    "CHGG",    "AI",    "NVDA",    "TSM",    "GOOGL",    "AMD",    
    "PAYO",     "LCID",    "PLUG",    "BYND",    "IBM",     "TM","SPY","CHGG","AI","NKLA","AMC","BYND" ,    "TDC", "INFA","SNOW",
     "PSTG","MDB",
             #expanded list of clean energy companies      
     
     "FSLR",'ENPH','SEDG','ARRY','NXT','ENVX','MVST','EOSE','FLNC','EVGO','ITRI','AMSC','POWI','VICR','NVTS','CLNE','GEVO','MNTK','ELVA','XEL','AEP','RNW',"INTC",
     "ARQQ","MU","SMCI",
       
       #balancing tech heavy
       "TRV","PGR","BHP","COST","MRK","NFLX","RMBS","ALB","VZ","SLDPW","AAPL","PG","ROP"
       
       
       
        
        
           ]
START_ISO = "2020-01-01T00:00:00Z"  # as far back as possible
from datetime import datetime, timedelta, timezone

def last_weekday_utc_string() -> str:
    now = datetime.now(timezone.utc)
    # Mon=0 … Sun=6  → subtract {Mon:3, Tue–Sat:1, Sun:2}
    days_back = [3, 1, 1, 1, 1, 1, 2][now.weekday()]
    dt = (now - timedelta(days=days_back)).replace(hour=0, minute=0, second=0, microsecond=0)
    return dt.strftime("%Y-%m-%dT%H:%M:%SZ")

END_ISO = last_weekday_utc_string()
END_ISO

'2025-10-17T00:00:00Z'

In [14]:
for sym in symbols:
    build_symbol_csv(sym, START_ISO, END_ISO, out_dir="./data", threshold=0.001)

✅ ./data\IBM_30s.csv already exists. Skipping API fetch.
✅ ./data\RGTI_30s.csv already exists. Skipping API fetch.
✅ ./data\QBTS_30s.csv already exists. Skipping API fetch.
✅ ./data\QUBT_30s.csv already exists. Skipping API fetch.
✅ ./data\IONQ_30s.csv already exists. Skipping API fetch.
✅ ./data\QS_30s.csv already exists. Skipping API fetch.
✅ ./data\AMD_30s.csv already exists. Skipping API fetch.
✅ ./data\SLDP_30s.csv already exists. Skipping API fetch.
✅ ./data\MSFT_30s.csv already exists. Skipping API fetch.
✅ ./data\CHGG_30s.csv already exists. Skipping API fetch.
✅ ./data\AI_30s.csv already exists. Skipping API fetch.
✅ ./data\NVDA_30s.csv already exists. Skipping API fetch.
✅ ./data\TSM_30s.csv already exists. Skipping API fetch.
✅ ./data\GOOGL_30s.csv already exists. Skipping API fetch.
✅ ./data\AMD_30s.csv already exists. Skipping API fetch.
✅ ./data\PAYO_30s.csv already exists. Skipping API fetch.
✅ ./data\LCID_30s.csv already exists. Skipping API fetch.
✅ ./data\PLUG_30s.csv

C:\Users\brobi\AppData\Local\Temp\ipykernel_22460\3929963377.py:29: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_30s = df.resample("30S").ffill()


📈 Adding indicators and target for TDC...
💾 Saved TDC → ./data\TDC_30s.csv
✅ ./data\INFA_30s.csv already exists. Skipping API fetch.
✅ ./data\SNOW_30s.csv already exists. Skipping API fetch.
✅ ./data\PSTG_30s.csv already exists. Skipping API fetch.
✅ ./data\MDB_30s.csv already exists. Skipping API fetch.
📥 Fetching 1-min IEX data for FSLR...
⏳ Resampling FSLR to 30-second bars...


C:\Users\brobi\AppData\Local\Temp\ipykernel_22460\3929963377.py:29: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_30s = df.resample("30S").ffill()


📈 Adding indicators and target for FSLR...
💾 Saved FSLR → ./data\FSLR_30s.csv
✅ ./data\ENPH_30s.csv already exists. Skipping API fetch.
✅ ./data\SEDG_30s.csv already exists. Skipping API fetch.
✅ ./data\ARRY_30s.csv already exists. Skipping API fetch.
✅ ./data\NXT_30s.csv already exists. Skipping API fetch.
✅ ./data\ENVX_30s.csv already exists. Skipping API fetch.
✅ ./data\MVST_30s.csv already exists. Skipping API fetch.
✅ ./data\EOSE_30s.csv already exists. Skipping API fetch.
✅ ./data\FLNC_30s.csv already exists. Skipping API fetch.
✅ ./data\EVGO_30s.csv already exists. Skipping API fetch.
✅ ./data\ITRI_30s.csv already exists. Skipping API fetch.
✅ ./data\AMSC_30s.csv already exists. Skipping API fetch.
✅ ./data\POWI_30s.csv already exists. Skipping API fetch.
✅ ./data\VICR_30s.csv already exists. Skipping API fetch.
✅ ./data\NVTS_30s.csv already exists. Skipping API fetch.
✅ ./data\CLNE_30s.csv already exists. Skipping API fetch.
✅ ./data\GEVO_30s.csv already exists. Skipping API fe

...

📈 Adding indicators and target for AAPL...
💾 Saved AAPL → ./data\AAPL_30s.csv
📥 Fetching 1-min IEX data for PG...
⏳ Resampling PG to 30-second bars...


C:\Users\brobi\AppData\Local\Temp\ipykernel_22460\3929963377.py:29: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_30s = df.resample("30S").ffill()


📈 Adding indicators and target for PG...
💾 Saved PG → ./data\PG_30s.csv
📥 Fetching 1-min IEX data for ROP...
⏳ Resampling ROP to 30-second bars...


C:\Users\brobi\AppData\Local\Temp\ipykernel_22460\3929963377.py:29: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_30s = df.resample("30S").ffill()


📈 Adding indicators and target for ROP...
💾 Saved ROP → ./data\ROP_30s.csv


In [15]:
def build_symbol_csv(symbol: str, start_iso: str, end_iso: str | None = None,
                     out_dir: str = "./data", threshold: float = 0.001):
    os.makedirs(out_dir, exist_ok=True)
    csv_path = os.path.join(out_dir, f"{symbol}_30s.csv")

    # Resolve start/end with a safety buffer on end (avoid super-recent SIP minutes)
    start = pd.Timestamp(start_iso, tz="UTC").to_pydatetime()
    end = pd.Timestamp(end_iso, tz="UTC").to_pydatetime() if end_iso else None
    end = _safe_end(end)  # utc-aware

    # --- If no file: full build exactly like before ---
    if not os.path.exists(csv_path):
        print(f"📥 [full] Fetching 1-min IEX data for {symbol}...")
        raw = fetch_1m_bars(symbol, start, end)
        if raw.empty:
            print(f"⚠️ No data for {symbol}")
            return

        print(f"⏳ Resampling {symbol} to 30-second bars...")
        bars_30s = resample_to_30s(raw)

        print(f"📈 Adding indicators and target for {symbol}...")
        final_df = add_indicators_and_target(bars_30s, threshold)

        final_df.to_csv(csv_path, index=True)
        print(f"💾 Saved {symbol} → {csv_path}")
        return final_df

    # --- If file exists: incremental append ---
    print(f"🔎 Found existing file → {csv_path}. Checking for new data to append…")
    existing = pd.read_csv(csv_path, parse_dates=["timestamp"], index_col="timestamp")
    # ensure tz-aware UTC index
    if existing.index.tz is None:
        existing.index = existing.index.tz_localize("UTC")
    else:
        existing.index = existing.index.tz_convert("UTC")

    last_ts = existing.index.max()  # last 30s bar we have (UTC)
    # Compute the "safe" ET day we can update through (based on end)
    end_et_date = pd.Timestamp(end).tz_convert("America/New_York").date()
    last_et_date = pd.Timestamp(last_ts).tz_convert("America/New_York").date()

    if last_et_date >= end_et_date:
        print(f"✅ Up to date through {last_et_date}. No fetch needed.")
        return existing

    # For robust resample and rolling continuity, start slightly before last_ts
    # (overlap 30 minutes avoids any boundary artifacts)
    overlap = pd.Timedelta(minutes=30)
    inc_start = (last_ts - overlap).to_pydatetime()

    print(f"📥 [incremental] {symbol}: fetching new 1-min bars from {inc_start} → {end} (UTC)")
    raw_inc = fetch_1m_bars(symbol, inc_start, end)
    if raw_inc.empty:
        print(f"ℹ️ No new minute bars returned. Keeping file as-is.")
        return existing

    # Convert to 30s and merge with existing
    print(f"⏳ Resampling incremental chunk to 30-second bars…")
    bars_30s_inc = resample_to_30s(raw_inc)

    # Build a unified base of O/H/L/C/V, then recompute indicators over the whole set
    base_cols = ["open", "high", "low", "close", "volume"]
    # ensure existing has the base cols (it should, from your earlier save)
    if not set(base_cols).issubset(existing.columns):
        # if not, keep whatever overlap exists
        existing_base = existing.reindex(columns=base_cols).dropna(how="all")
    else:
        existing_base = existing[base_cols].copy()

    merged_base = (
        pd.concat([existing_base, bars_30s_inc[base_cols]])
          .sort_index()
          .drop_duplicates()  # index (timestamp) dupes collapse
    )

    print("📈 Recomputing indicators/target for continuity…")
    final_df = add_indicators_and_target(merged_base, threshold)

    final_df.to_csv(csv_path, index=True)
    print(f"✅ Appended & saved → {csv_path} (rows={len(final_df)})")
    return final_df


In [16]:
for sym in symbols:
    build_symbol_csv(sym, START_ISO, END_ISO, out_dir="./data", threshold=0.001)

🔎 Found existing file → ./data\IBM_30s.csv. Checking for new data to append…
✅ Up to date through 2025-10-16. No fetch needed.
🔎 Found existing file → ./data\RGTI_30s.csv. Checking for new data to append…
✅ Up to date through 2025-10-16. No fetch needed.
🔎 Found existing file → ./data\QBTS_30s.csv. Checking for new data to append…
✅ Up to date through 2025-10-16. No fetch needed.
🔎 Found existing file → ./data\QUBT_30s.csv. Checking for new data to append…
✅ Up to date through 2025-10-16. No fetch needed.
🔎 Found existing file → ./data\IONQ_30s.csv. Checking for new data to append…
✅ Up to date through 2025-10-16. No fetch needed.
🔎 Found existing file → ./data\QS_30s.csv. Checking for new data to append…
✅ Up to date through 2025-10-16. No fetch needed.
🔎 Found existing file → ./data\AMD_30s.csv. Checking for new data to append…
✅ Up to date through 2025-10-16. No fetch needed.
🔎 Found existing file → ./data\SLDP_30s.csv. Checking for new data to append…
✅ Up to date through 2025-10-1

C:\Users\brobi\AppData\Local\Temp\ipykernel_22460\3929963377.py:29: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_30s = df.resample("30S").ffill()


✅ Appended & saved → ./data\NKLA_30s.csv (rows=232279)
🔎 Found existing file → ./data\AMC_30s.csv. Checking for new data to append…
✅ Up to date through 2025-10-16. No fetch needed.
🔎 Found existing file → ./data\BYND_30s.csv. Checking for new data to append…
✅ Up to date through 2025-10-16. No fetch needed.
🔎 Found existing file → ./data\TDC_30s.csv. Checking for new data to append…
✅ Up to date through 2025-10-16. No fetch needed.
🔎 Found existing file → ./data\INFA_30s.csv. Checking for new data to append…
✅ Up to date through 2025-10-16. No fetch needed.
🔎 Found existing file → ./data\SNOW_30s.csv. Checking for new data to append…
✅ Up to date through 2025-10-16. No fetch needed.
🔎 Found existing file → ./data\PSTG_30s.csv. Checking for new data to append…
✅ Up to date through 2025-10-16. No fetch needed.
🔎 Found existing file → ./data\MDB_30s.csv. Checking for new data to append…
✅ Up to date through 2025-10-16. No fetch needed.
🔎 Found existing file → ./data\FSLR_30s.csv. Checkin